In [ ]:
%pip install numpy pandas scikit-learn xgboost sweetviz matplotlib seaborn

In [ ]:
# libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)
from xgboost import XGBClassifier


In [ ]:
#default = pd.read_csv("../Data/creditcard.csv")
default = pd.read_csv("../Data/creditcard.csv")


In [ ]:
#use sweetviz to explore the data
report = sv.analyze(default)
report.show_html()



In [ ]:
TARGET = "default payment next month"
X = default.drop(columns=[TARGET])
y = default[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True
)

# Class imbalance: up-weight the minority positive class (default == 1)
# RF: sklearn balanced class weights. XGB: scale_pos_weight ≈ n_negative / n_positive on training labels
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

rf_clf = RandomForestClassifier(random_state=42, class_weight="balanced")
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
y_proba_rf = rf_clf.predict_proba(X_test)[:, 1]

print("RandomForestClassifier (class_weight='balanced')")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba_rf):.4f}")
print(classification_report(y_test, y_pred_rf, digits=4))

xgb_clf = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
)
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
y_proba_xgb = xgb_clf.predict_proba(X_test)[:, 1]

print(f"XGBClassifier (scale_pos_weight={scale_pos_weight:.3f})")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba_xgb):.4f}")
print(classification_report(y_test, y_pred_xgb, digits=4))

In [ ]:
# Cross-val clones rf_clf / xgb_clf hyperparameters (balanced RF, XGB scale_pos_weight from cell above)
rf_cv = cross_val_score(rf_clf, X, y, cv=5, scoring="roc_auc")
xgb_cv = cross_val_score(xgb_clf, X, y, cv=5, scoring="roc_auc")

print(f"RandomForestClassifier (balanced, CV ROC-AUC): {rf_cv.mean():.4f} ± {rf_cv.std() * 2:.4f}")
print(f"XGBClassifier (scale_pos_weight, CV ROC-AUC): {xgb_cv.mean():.4f} ± {xgb_cv.std() * 2:.4f}")

#each fold in cv test value
print(rf_cv)
print(xgb_cv)

The models performed decently well however they do well on the majority class, which is individuals who don't commit credit card fraud. In a apllication like this, it is almost certain that this model would be useless, we need to weight missing fraud cases more heavily.

In [ ]:
# Baseline models: 5-fold cross-validation with recall (positive class = default)
# Same imbalance handling as above; scale_pos_weight from full y (common CV heuristic)

scale_pos_weight_cv = float((y == 0).sum() / (y == 1).sum())
rf_base = RandomForestClassifier(random_state=42, class_weight="balanced")
xgb_base = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight_cv,
)

cv = 5
rf_recall_cv = cross_val_score(rf_base, X, y, cv=cv, scoring="recall")
xgb_recall_cv = cross_val_score(xgb_base, X, y, cv=cv, scoring="recall")

print("RandomForestClassifier (balanced class_weight) — recall by fold:", rf_recall_cv)
print(
    f"  Mean recall: {rf_recall_cv.mean():.4f} ± {rf_recall_cv.std() * 2:.4f} (2×std)\n"
)

print("XGBClassifier (scale_pos_weight) — recall by fold:", xgb_recall_cv)
print(
    f"  Mean recall: {xgb_recall_cv.mean():.4f} ± {xgb_recall_cv.std() * 2:.4f} (2×std)"
)


In [ ]:
# XGBoost with row subsample 0.8 (each tree uses ~80% of rows; same imbalance settings as cell 4)
# Recompute scale_pos_weight here so this cell runs even if cell 4 was not executed (needs y_train)
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

xgb_clf_sub = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
)
xgb_clf_sub.fit(X_train, y_train)
y_pred_xgb_sub = xgb_clf_sub.predict(X_test)
y_proba_xgb_sub = xgb_clf_sub.predict_proba(X_test)[:, 1]

print("XGBClassifier (subsample=0.8, scale_pos_weight as above)")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb_sub):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba_xgb_sub):.4f}")
print(classification_report(y_test, y_pred_xgb_sub, digits=4))


In [ ]:
print("Starting Random Forest tuning...\n")

rf_grid_base = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Smaller, faster grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10],
    "min_samples_leaf": [1, 4],
}

rf_grid_search = GridSearchCV(
    rf_grid_base,
    param_grid,
    cv=3,              # faster (was 5)
    scoring="roc_auc",
    n_jobs=1,
    refit=True,
    verbose=3          # clearer progress
)

rf_grid_search.fit(X_train, y_train)

print("\nTuning Complete\n")

rf_clf_tuned = rf_grid_search.best_estimator_
y_pred_rf_tuned = rf_clf_tuned.predict(X_test)
y_proba_rf_tuned = rf_clf_tuned.predict_proba(X_test)[:, 1]

print("Best params:", rf_grid_search.best_params_)
print(f"Best CV ROC-AUC: {rf_grid_search.best_score_:.4f}")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_rf_tuned):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_test, y_proba_rf_tuned):.4f}")
print(classification_report(y_test, y_pred_rf_tuned, digits=4))

In [ ]:
# XGBoost: brief tuning over 3 hyperparameters
# I am choosing to KEEP the class imbalance adjustment (scale_pos_weight)
# and tune n_estimators, max_depth, and learning_rate.

# Baseline XGBoost for comparison
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
)

xgb_base.fit(X_train, y_train)
y_pred_xgb_base = xgb_base.predict(X_test)
y_proba_xgb_base = xgb_base.predict_proba(X_test)[:, 1]

base_test_auc = roc_auc_score(y_test, y_proba_xgb_base)
base_test_acc = accuracy_score(y_test, y_pred_xgb_base)

print("Baseline XGBoost")
print(f"Test Accuracy: {base_test_acc:.4f}")
print(f"Test ROC-AUC:  {base_test_auc:.4f}")
print(classification_report(y_test, y_pred_xgb_base, digits=4))


# Tuned XGBoost
xgb_grid_base = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
)

xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.10, 0.20],
}

xgb_grid_search = GridSearchCV(
    estimator=xgb_grid_base,
    param_grid=xgb_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True,
)

xgb_grid_search.fit(X_train, y_train)

xgb_clf_tuned = xgb_grid_search.best_estimator_
y_pred_xgb_tuned = xgb_clf_tuned.predict(X_test)
y_proba_xgb_tuned = xgb_clf_tuned.predict_proba(X_test)[:, 1]

tuned_test_auc = roc_auc_score(y_test, y_proba_xgb_tuned)
tuned_test_acc = accuracy_score(y_test, y_pred_xgb_tuned)

print("\nTuned XGBoost")
print("Best params:", xgb_grid_search.best_params_)
print(f"Best CV ROC-AUC: {xgb_grid_search.best_score_:.4f}")
print(f"Test Accuracy:   {tuned_test_acc:.4f}")
print(f"Test ROC-AUC:    {tuned_test_auc:.4f}")
print(classification_report(y_test, y_pred_xgb_tuned, digits=4))

# Compare baseline vs tuned
print("\nComparison")
print(f"Accuracy change: {tuned_test_acc - base_test_acc:+.4f}")
print(f"ROC-AUC change:  {tuned_test_auc - base_test_auc:+.4f}")

Model performance increased after tuning XGBoost. ROC-AUC improved from 0.7604 to 0.7795, and recall for the minority class also increased. Accuracy decreased slightly, but this is expected when improving performance on an imbalanced dataset

Random Forest performed better out-of-the-box, with Accuracy = 0.8154 and ROC-AUC = 0.7625, compared to XGBoost (Accuracy = 0.7646, ROC-AUC = 0.7604), But after tuning XGBoost achieved the best ROC-AUC (0.7795).

In [ ]:
import nbformat

file = "Inclass_04_09.ipynb"

nb = nbformat.read(file, as_version=4)

if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

nbformat.write(nb, file)

print("Notebook cleaned successfully")

In [ ]:
%pip install nbformat

In [ ]:
import nbformat

file = "Inclass_04_09.ipynb"

nb = nbformat.read(file, as_version=4)

if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

nbformat.write(nb, file)

print("Notebook cleaned successfully")